In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/FF++.zip'
extract_path = '/content/FF_plus_plus_extracted'

if not os.path.exists(extract_path):
    print("[INFO] Extracting FF++.zip (this will take a few minutes)...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("[INFO] Extraction complete.")
else:
    print("[INFO] Dataset already extracted.")


[INFO] Extracting FF++.zip (this will take a few minutes)...
[INFO] Extraction complete.


In [ ]:
import glob

real_dir = os.path.join(extract_path, 'real')
fake_dir = os.path.join(extract_path, 'fake')

real_videos = sorted(glob.glob(real_dir + "/*.mp4"))
fake_videos = sorted(glob.glob(fake_dir + "/*.mp4"))

# Support for .avi, .mov
real_videos += sorted(glob.glob(real_dir + "/*.avi"))
real_videos += sorted(glob.glob(real_dir + "/*.mov"))
fake_videos += sorted(glob.glob(fake_dir + "/*.avi"))
fake_videos += sorted(glob.glob(fake_dir + "/*.mov"))

video_paths = real_videos + fake_videos
labels = [0] * len(real_videos) + [1] * len(fake_videos)

print(f"Found {len(real_videos)} real and {len(fake_videos)} fake videos.")


Found 200 real and 200 fake videos.


In [ ]:
from sklearn.model_selection import train_test_split

train_paths, test_paths, train_labels, test_labels = train_test_split(
    video_paths, labels, test_size=0.2, stratify=labels, random_state=42)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels, test_size=0.2, stratify=train_labels, random_state=42)


In [ ]:
import cv2
import numpy as np
import tensorflow as tf

class VideoFrameGenerator(tf.keras.utils.Sequence):
    def __init__(self, video_files, labels, batch_size=8, frames_per_video=8, img_size=(224, 224), shuffle=True):
        self.video_files = video_files
        self.labels = labels
        self.batch_size = batch_size
        self.frames_per_video = frames_per_video
        self.img_size = img_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.video_files))
        self.on_epoch_end()
    def __len__(self):
        return int(np.ceil(len(self.video_files) / self.batch_size))
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)
    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_files = [self.video_files[i] for i in batch_indices]
        batch_labels = [self.labels[i] for i in batch_indices]
        X, y = [], []
        for file, label in zip(batch_files, batch_labels):
            frames = self.sample_frames(file)
            X.append(frames)
            y.append(label)
        return np.array(X), np.array(y)
    def sample_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_ids = np.linspace(0, n_frames-1, self.frames_per_video, dtype=np.int32) if n_frames > 0 else []
        frames = []
        for fid in frame_ids:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fid)
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, self.img_size)
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame/255.0)
            else:
                frames.append(np.zeros((*self.img_size, 3)))
        cap.release()
        # shape: (frames_per_video, H, W, 3)
        frames = np.array(frames)
        return frames


In [ ]:
batch_size = 8
frames_per_video = 8
img_size = (224, 224)

train_gen = VideoFrameGenerator(train_paths, train_labels, batch_size, frames_per_video, img_size)
val_gen = VideoFrameGenerator(val_paths, val_labels, batch_size, frames_per_video, img_size)
test_gen = VideoFrameGenerator(test_paths, test_labels, batch_size, frames_per_video, img_size, shuffle=False)


In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

input_shape = (frames_per_video, img_size[0], img_size[1], 3)
inputs = layers.Input(shape=input_shape)

# Process each frame independently using the same EfficientNet
cnn = EfficientNetB0(include_top=False, pooling='avg', weights='imagenet', input_shape=img_size+(3,))
frame_features = layers.TimeDistributed(cnn)(inputs)
x = layers.GlobalAveragePooling1D()(frame_features)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 8, 224, 224, 3) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 8, 1280)        │     4,049,571 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,050,852 (15.45 MB)

 Trainable params: 4,008,829 (15.29 MB)

 Non-trainable params: 42,023 (164.16 KB)

In [ ]:
callback = tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=[callback],
    verbose=1
)


Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 2206s 58s/step - accuracy: 0.4851 - loss: 0.8121 - val_accuracy: 0.5000 - val_loss: 0.7201
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 1772s 56s/step - accuracy: 0.6138 - loss: 0.6802 - val_accuracy: 0.5000 - val_loss: 0.7166
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 1866s 58s/step - accuracy: 0.6599 - loss: 0.6278 - val_accuracy: 0.5000 - val_loss: 0.7002
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 1843s 58s/step - accuracy: 0.6489 - loss: 0.6206 - val_accuracy: 0.5156 - val_loss: 83.0014
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 1774s 56s/step - accuracy: 0.6772 - loss: 0.5968 - val_accuracy: 0.5469 - val_loss: 0.6908
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 1724s 54s/step - accuracy: 0.7917 - loss: 0.4832 - val_accuracy: 0.5000 - val_loss: 261.7774
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 1730s 54s/step - accuracy: 0.7611 - loss: 0.4759 - val_accuracy: 0.5000 - val_loss: 0.9338
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 1743s 55s/step - accuracy: 0.7772 - loss: 0.4670 - val_a

In [ ]:
model.save('fake_video_detector_model.keras')
print("Model saved successfully!")

Model saved successfully!


In [10]:
loss, acc = model.evaluate(test_gen, verbose=1)
print(f"Test accuracy: {acc:.3f}")


NameError: name 'test_gen' is not defined

In [1]:
from tensorflow.keras.models import load_model

model = load_model('fake_video_detector_model.keras')
print("Model loaded successfully!")


Model loaded successfully!


In [2]:
import cv2
import numpy as np

def preprocess_video(video_path, frames_per_video=8, img_size=(224, 224)):
    cap = cv2.VideoCapture(video_path)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_ids = np.linspace(0, n_frames-1, frames_per_video, dtype=np.int32) if n_frames > 0 else []
    frames = []
    for fid in frame_ids:
        cap.set(cv2.CAP_PROP_POS_FRAMES, fid)
        ret, frame = cap.read()
        if ret:
            frame = cv2.resize(frame, img_size)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame / 255.0)
        else:
            frames.append(np.zeros((*img_size, 3)))
    cap.release()
    return np.expand_dims(np.array(frames), axis=0)  # Shape (1, frames, H, W, 3)


In [9]:
test_video_path = 'video1.mp4'  # Update with your video path
input_data = preprocess_video(test_video_path)

prediction = model.predict(input_data)[0][0]
print(f"Prediction score (0=Real, 1=Fake): {prediction:.4f}")

if prediction >= 0.5:
    print("Predicted: Fake video")
else:
    print("Predicted: Real video")


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Prediction score (0=Real, 1=Fake): 0.5231
Predicted: Fake video
